# Exercise: produce_01 — your first producer

**Goal:** Send a single event to Redpanda and verify it arrives.

**Use case:** IoT sensors in houses report electricity (`strom`) and water (`wasser`) usage.

> Stuck? Compare with `../02_demo/demo_produce.ipynb`. Full solution in `solutions/exercise_01_produce_single.ipynb`.

## Step 1 — imports and broker connection

**Question:** What is a *broker*? What does `bootstrap.servers` do?

In [ ]:
from confluent_kafka import Producer
import json, time

BROKER = 'redpanda:29092'   # internal listener in the docker network

conf = {
    'bootstrap.servers': BROKER,
    'client.id':         'python-producer',
}
producer = Producer(conf)
print(f'Producer connected to {BROKER}')

## Step 2 — build the event

An event has three parts:
- **Topic** — which category? (`strom` or `wasser`)
- **Key** — which house? (`haus_a`, `haus_b`, ...) — the key picks the partition
- **Value** — the sensor reading as JSON

**Task:** fill in the two missing values below.

In [ ]:
topic_name    = 'strom'
message_key   = 'haus_a'
message_value = json.dumps({
    'sensor':    'strom',
    'haus':      'haus_a',
    'wert':      None,        # TODO: pick a number, e.g. 42.5
    'einheit':   None,        # TODO: which unit fits electricity? ('kWh')
    'timestamp': time.time(),
})

print(f'Topic:  {topic_name}')
print(f'Key:    {message_key}')
print(f'Value:  {message_value}')

## Step 3 — send it

**Task:** call `producer.produce(...)` with the topic, key, value and callback.

```python
producer.produce(topic, key=..., value=..., callback=...)
# both key and value must be bytes — call .encode('utf-8') on the strings
```

**Then:** open the **Redpanda Console** (port `8080`) → topic `strom` → *Messages*. You should see partition, offset, key, value, timestamp.

In [ ]:
def delivery_report(err, msg):
    if err:
        print(f'Delivery failed: {err}')
    else:
        print(f'Delivered to {msg.topic()} [partition {msg.partition()}] offset {msg.offset()}')

# TODO: call producer.produce(...) with topic_name, message_key, message_value, delivery_report


# TODO: call producer.flush() — why is this needed?
print('Done.')

## Task A — second event with a different key

Send another event to `strom` with key `haus_b`. Check the Console: which partition did it land in?

> Hint: Kafka hashes the key to pick a partition. **Same key → always same partition.**

In [ ]:
# TODO: build message_value_b for haus_b and call producer.produce(...) + flush


## Task B — write to a different topic

Send a water-consumption event for `haus_a` to topic `wasser`. Use `'Liter'` as the unit.

In [ ]:
# TODO: send an event to topic 'wasser' with key 'haus_a' and a Liter value


## Task C — what happens when the broker is unreachable?

Run the cell below. It points the producer at a non-existent address.

**Questions:**
- What error message do you get?
- What does `message.timeout.ms` control?
- In production: what would you do with messages that fail to deliver? (Keyword: **Dead Letter Queue**)

In [ ]:
errors = []
def error_cb(err, msg):
    errors.append(str(err))

p_broken = Producer({
    'bootstrap.servers':  'localhost:9999',   # nothing listens here
    'message.timeout.ms': 4000,
    'socket.timeout.ms':  2000,
})
p_broken.produce('strom', key=b'haus_a', value=b'{"test": true}', callback=error_cb)
p_broken.flush(timeout=5)

if errors:
    print(f'Delivery failed (as expected): {errors[0]}')
else:
    print('Delivered (unexpected)')